# LoRA 微调：从原理到实践

LoRA（**Lo**w-**R**ank **A**daptation）是一种高效的大型预训练模型微调方法。

## 1. 为什么需要 LoRA？

全参数微调（Full Fine-tuning）存在两个主要问题：

- **显存开销巨大**：以 7B 参数的模型为例，全参数微调需要存储参数、梯度、优化器状态（如 Adam 的一阶矩和二阶矩），总显存约为参数量的 16-20 倍，单卡很难跑起来。
- **灾难性遗忘与部署成本**：每次下游任务都需要保存一份完整的模型副本，切换任务时必须重新加载巨大的权重文件。

LoRA 通过引入极少量可训练参数（通常只占原模型的 0.1%~1%），将显存需求降至原来的 1/3 左右，且可针对不同任务快速热插拔（只需切换很小的 LoRA 权重文件）。

## 2. 核心原理：低秩分解

LoRA 的灵感来自一个观察：模型在适配下游任务时，权重的更新矩阵通常是**低秩**的。也就是说，虽然权重矩阵 $W$ 维度很高（例如 $d \times k$），但它的变化量 $\Delta W$ 可以用两个更小的矩阵相乘来近似。

数学上，对于一个全连接层的权重 $W_0 \in \mathbb{R}^{d \times k}$，LoRA 将前向传播修改为：

$$h = W_0 x + \underbrace{\frac{\alpha}{r} \cdot B A x}_{\Delta W x}$$

其中：
- $A \in \mathbb{R}^{r \times k}$ 和 $B \in \mathbb{R}^{d \times r}$ 是新引入的可训练矩阵，且 $r \ll \min(d, k)$。
- $r$ 称为**秩**（rank），常用的值为 8、16、32。
- $\alpha$ 是一个缩放因子，通常与 $r$ 成比例，作用是调控 LoRA 的更新强度，防止初始阶段噪声过大。
- $W_0$ 完全冻结，不参与梯度更新。

初始化时，$A$ 通常使用高斯随机初始化，$B$ 初始化为全零矩阵。这样在最开始，$\Delta W = 0$，模型行为与原模型完全一致，随着训练逐渐学习偏移量。

### 可训练参数量分析

假设 $d = k = 4096$，原始参数有 $4096 \times 4096 \approx 16.7M$ 个。若设置秩 $r=16$，则：

$$\text{LoRA 参数} = 4096 \times 16 + 16 \times 4096 = 131,072 \approx 0.13M$$

参数量不到原来的 0.8%，这就是 LoRA 高效的来源。

## 3. LoRA 通常应用在哪些层？

在 Transformer 架构中，LoRA 最常加在**自注意力层的 Query（Q）和 Value（V）投影矩阵**上。偶尔也会加在 Key（K）、Output 投影、前馈网络（FFN）层。实践证明，只对 Q、V 添加 LoRA 已经足够达到很好效果，且效率最高。

对于 LLM（如 LLaMA、GPT），通常对 `q_proj` 和 `v_proj` 应用 LoRA。

## 4. LoRA 的主要优势

- **显存大幅降低**：不需要存储冻结参数的梯度和优化器状态，只需要为小矩阵 $A, B$ 存储这些。
- **训练速度更快**：反向传播只需要通过低秩矩阵，计算量小。
- **多任务共享基座**：可以训练多个任务的 LoRA 权重，共享同一个基座模型，推理时动态加载不同 LoRA，不增加额外推理延迟（可将 $BA$ 合并到原权重中）。
- **无推理延迟**：由于 $BA$ 可以合并到 $W_0$ 中得到一个新的权重矩阵 $W = W_0 + \frac{\alpha}{r} BA$，推理时结构与原模型完全一致，不会增加任何计算开销。
- **缓解灾难性遗忘**：基座权重不更新，保留了原有知识。

## 5. 动手教程：使用 PEFT 库微调 LLaMA 类模型

以 **Hugging Face 的 PEFT**（Parameter-Efficient Fine-Tuning）库 + Transformers 为例，对一个小型对话模型进行 LoRA 微调。整个过程可以在单张 16GB 显存的 GPU（如 T4、RTX 3060）上完成。

### 环境准备

In [ ]:
# 安装依赖
# pip install transformers accelerate peft datasets bitsandbytes scipy sentencepiece
# bitsandbytes 用于 4bit 量化加载，进一步节省显存

### 5.1 加载模型与分词器（使用 4bit 量化加载）

使用 `bitsandbytes` 的 `NF4` 量化，大幅降低基座模型的显存占用。

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

# 量化配置：4bit NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # 计算时使用 bfloat16
    bnb_4bit_use_double_quant=True,
)

# 使用 TinyLlama-1.1B 作为示例模型（仅 1.1B 参数，显存占用极低）
# 其他可选小模型：
#   - "Qwen/Qwen2.5-0.5B-Instruct" (0.5B)
#   - "TinyLlama/TinyLlama-1.1B-Chat-v1.0" (1.1B)
#   - "microsoft/phi-1_5" (1.3B)
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token   # 设置 pad token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

### 5.2 让模型适配 PEFT 训练

`prepare_model_for_kbit_training` 会在某些层插入梯度检查点，并转换某些模块以便梯度传播。

In [ ]:
model = prepare_model_for_kbit_training(model)

### 5.3 配置 LoRA

设置 `LoraConfig`：指定秩 `r`、alpha、dropout、以及要应用的模块名（用正则匹配）。

In [ ]:
lora_config = LoraConfig(
    r=8,                      # 秩（小模型可以用更小的 r）
    lora_alpha=16,            # alpha 缩放，通常为 r 的 2 倍
    target_modules=["q_proj", "v_proj"],  # TinyLlama 的注意力 Q/V
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

输出类似（TinyLlama-1.1B 为例）：
```
trainable params: 524,288 || all params: 1,103,237,120 || trainable%: 0.0475
```
可以看到只有约 0.05% 的参数可训练，显存占用极低。

### 5.4 准备数据集

以指令微调为例，需要将数据格式化成 prompt + response 形式。

In [ ]:
# 假设你的数据格式为 {"instruction":..., "output":...}
dataset = load_dataset("json", data_files="train.jsonl")

def format_example(example):
    # TinyLlama 使用 LLAMA 风格的对话模板
    prompt = f"<|user|>{example['instruction']}</s><|assistant|>"
    full_text = prompt + example["output"] + "</s>"
    return {"text": full_text}

dataset = dataset.map(format_example)

def tokenize_function(examples):
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,        # 小模型用较短的序列长度
        padding="max_length",
    )
    # 让 labels 等于 input_ids（自回归语言模型的损失）
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

### 5.5 训练参数与 Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-qwen",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # 实际 batch size = 4*4 = 16
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,                       # 如果 GPU 不支持 bf16 就开启 fp16
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",                # 不上报 wandb 等
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

### 5.6 保存与加载 LoRA 权重

训练完成后，保存 LoRA adapter：

In [ ]:
# 保存 LoRA adapter
model.save_pretrained("./lora-adapter-final")
tokenizer.save_pretrained("./lora-adapter-final")

推理时加载：

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, "./lora-adapter-final")

# 生成回答（使用 TinyLlama 的对话格式）
inputs = tokenizer("<|user|>你好，你是谁？</s><|assistant|>", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

### 5.7 合并 LoRA 权重并导出为完整模型（可选）

如果想省去推理时加载 adapter 的麻烦，可以将 LoRA 合并到基座权重中：

In [ ]:
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./merged-model")
tokenizer.save_pretrained("./merged-model")

## 6. 超参数调优建议

| 超参数 | 建议值 | 说明 |
|--------|--------|------|
| **秩 `r`** | 8~64 | 对于简单任务 `r=8` 足够；复杂任务可使用 16~64。一般 `r=16` 是性价比较高的选择。 |
| **`alpha`** | `r` 的 1~2 倍 | 通常设为 `r` 的 1~2 倍。增大 alpha 意味着给 LoRA 更新更大的幅度，但需要配合学习率调整。 |
| **`target_modules`** | `q_proj`, `v_proj` | 对于 LLaMA/Qwen/GPT-NeoX 等，建议至少包含 `q_proj` 和 `v_proj`。加入更多模块可提升性能，但参数量也会增加。 |
| **LoRA dropout** | 0.0~0.1 | 用于防止过拟合，小数据时可略设高一点。 |
| **学习率** | 1e-4 ~ 5e-4 | LoRA 通常需要比全参数微调稍大的学习率。 |

## 7. 常见问题

**Q：LoRA 可以和其他 PEFT 方法叠加吗？**  
A：可以。例如同时使用 **QLoRA**（量化 + LoRA），即本文演示的方式，通过 4bit 量化基座模型，在量化后的模型上应用 LoRA，进一步降低显存。

**Q：LoRA 权重可以跨模型吗？**  
A：不行。LoRA 权重与基座模型的结构和维度绑定，不同模型结构不通用。但相同架构不同参数量的模型也可能不兼容。

**Q：LoRA 训练后是否必须合并权重？**  
A：不必。你可以保留基座模型不变，动态加载不同的 LoRA adapter，实现多任务快速切换。推理时如果不合并，adapter 会带来微小的额外计算开销（需额外计算 $BAx$ 再加到原输出）。通常推理框架（如 vLLM、Text Generation Inference）支持直接加载 adapter，无需手动合并。

## 8. 总结

LoRA 微调的核心就是：**冻结大模型，只训练低秩矩阵**。这个简单而强大的思想使得我们在消费级 GPU 上也能微调 7B、13B 甚至更大的模型。配合量化技术（QLoRA），更能在有限硬件上实现高效定制化。

动手实践时，PEFT 库已经提供了非常完善的接口，你只需要指定 `target_modules` 和秩 `r`，即可轻松将 LoRA 注入模型。